# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset on Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata and inspect basic information
ds = mlc.Dataset(croissant_url)
md = ds.metadata

print(f"Dataset Name: {md.name}\n")
print(f"Description: {md.description}\n")
print(f"Identifier: {md.identifier}")
print(f"License: {md.license}")

## 2. Data Overview
Review available **record sets**, their fields, and their Croissant `@id`s.

We enumerate all record sets and fields in this dataset using the Croissant API. All references below are via `@id`.

In [ ]:
# List all record sets and their field @ids
record_sets = ds.list_record_sets()
print("All record sets (@id):")
pprint.pprint(record_sets)

# For each record set, show its field @ids
record_set_fields = {}
for rs_id in record_sets:
    fields = ds.list_fields(record_set=rs_id)
    record_set_fields[rs_id] = fields
    print(f"\nRecord Set {rs_id} fields (@id):")
    pprint.pprint(fields)

# Display a sample record from each record set
for rs_id in record_sets:
    print(f"\nSample record from record set {rs_id}:")
    try:
        rec = next(ds.records(record_set=rs_id))
        pprint.pprint(rec)
    except StopIteration:
        print("(No records)")

## 3. Data Extraction
Load data from each record set into a separate DataFrame. 

All record set, field, and column references are by their Croissant `@id`.

In [ ]:
# Collect all record set @ids
rs_ids = ds.list_record_sets()
dataframes = {}

for rs_id in rs_ids:
    # Load all records for this record set
    records = list(ds.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for record set {rs_id}")

# For demonstration, select the first (main) record set
main_rs_id = rs_ids[0] if rs_ids else None
if main_rs_id and not dataframes[main_rs_id].empty:
    print(f"\nRecord set columns (@id) for {main_rs_id}:\n{list(dataframes[main_rs_id].columns)}\n")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Here we demonstrate how to process and analyze the numeric fields: filter by value, normalize, and group/categorize by key attributes.

All field and record set references use their Croissant `@id`.

In [ ]:
# Select a numeric field and a categorical/grouping field by their @id as shown above

# Example: suppose fields
#  - Patient age: '@id': 'age'      (numeric)
#  - Sex:         '@id': 'sex'      (categorical)

# Please replace with the actual @id values for your dataset.
numeric_field_id = None
group_field_id = None
if main_rs_id:
    # Try to auto-detect numeric and group fields
    cols = list(dataframes[main_rs_id].columns)
    lower_cols = [c.lower() for c in cols]
    # Guess 'age' (or similar) as numeric, 'sex' or 'gender' as group
    for c in cols:
        if 'age' in c.lower():
            numeric_field_id = c
        if 'sex' in c.lower() or 'gender' in c.lower():
            group_field_id = c

    print(f"Selected numeric field (@id): {numeric_field_id}")
    print(f"Selected group field (@id): {group_field_id}")

    # If numeric field is found and contains numeric data
    df = dataframes[main_rs_id].copy()
    if numeric_field_id and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        threshold = df[numeric_field_id].mean()  # Example: threshold = mean
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by key attribute if available
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
            display(grouped_df.head())
    else:
        print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize the distribution of selected numeric and group fields.

All columns and field references are by their Croissant `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and numeric_field_id and numeric_field_id in dataframes[main_rs_id]:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[main_rs_id][numeric_field_id].dropna(), kde=True, bins=10)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in dataframes[main_rs_id]:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=dataframes[main_rs_id][group_field_id], y=dataframes[main_rs_id][numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to explore and process the FAIR² colorectal cancer dataset using `mlcroissant`.

- All operations referenced fields, record sets, and columns by their Croissant `@id`.
- We loaded, visualized, and analyzed clinical and molecular patient-level data.
- The notebook can be adapted by updating the field `@id`s to target specific analysis needs.

**Key findings:**
- Basic distributional analysis and grouping by key attributes (e.g., sex/gender, age) is supported.
- For more complex exploration, consult the `mlcroissant` [documentation](https://github.com/mlcommons/croissant) and refer to the dataset schema for further `@id` references.